In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# =========================================================
# DATABASE GOLD
# =========================================================

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# =========================================================
# DIM_CLIENTE
# =========================================================

clientes = spark.table("silver.Cliente")

dim_cliente = (
    clientes
    .select(
        monotonically_increasing_id().alias("sk_cliente"),

        col("id_cliente").alias("id_cliente_origem"),

        col("nome_cliente"),
        col("segmento"),
        col("cidade"),
        col("estado"),

        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("RS","SC","PR"), "Sul")
        .when(col("estado").isin("BA","PE","CE","MA","PB","RN","AL","SE","PI"), "Nordeste")
        .when(col("estado").isin("GO","MT","MS","DF"), "Centro-Oeste")
        .when(col("estado").isin("AM","PA","RO","RR","AP","AC","TO"), "Norte")
        .otherwise("Outros")
        .alias("regiao"),

        col("data_cadastro")
    )
)

dim_cliente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_cliente")

# =========================================================
# DIM_CENTRO
# =========================================================

centros = spark.table("silver.CentroDistribuicao")

dim_centro = (
    centros
    .select(
        monotonically_increasing_id().alias("sk_centro"),

        col("id_centro").alias("id_centro_origem"),

        col("nome_centro"),
        col("cidade"),
        col("estado"),

        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("RS","SC","PR"), "Sul")
        .when(col("estado").isin("BA","PE","CE","MA","PB","RN","AL","SE","PI"), "Nordeste")
        .when(col("estado").isin("GO","MT","MS","DF"), "Centro-Oeste")
        .when(col("estado").isin("AM","PA","RO","RR","AP","AC","TO"), "Norte")
        .otherwise("Outros")
        .alias("regiao"),

        col("capacidade_operacional")
    )
)

dim_centro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_centro")

# =========================================================
# DIM_VEICULO
# =========================================================

veiculos = spark.table("silver.Veiculo")

dim_veiculo = (
    veiculos
    .select(
        monotonically_increasing_id().alias("sk_veiculo"),

        col("id_veiculo").alias("id_veiculo_origem"),

        col("placa"),
        col("modelo"),
        col("tipo_veiculo"),
        col("capacidade_kg"),

        when(col("capacidade_kg") < 2000, "Leve")
        .when(col("capacidade_kg") < 10000, "Medio")
        .otherwise("Pesado")
        .alias("faixa_capacidade"),

        col("ano_fabricacao"),
        col("status_veiculo")
    )
)

dim_veiculo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_veiculo")

# =========================================================
# DIM_MOTORISTA
# =========================================================

motoristas = spark.table("silver.Motorista")

dim_motorista = (
    motoristas
    .select(
        monotonically_increasing_id().alias("sk_motorista"),

        col("id_motorista").alias("id_motorista_origem"),

        col("nome_motorista"),
        col("cidade"),
        col("estado"),

        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("RS","SC","PR"), "Sul")
        .when(col("estado").isin("BA","PE","CE","MA","PB","RN","AL","SE","PI"), "Nordeste")
        .when(col("estado").isin("GO","MT","MS","DF"), "Centro-Oeste")
        .when(col("estado").isin("AM","PA","RO","RR","AP","AC","TO"), "Norte")
        .otherwise("Outros")
        .alias("regiao"),

        col("categoria_cnh"),
        col("data_admissao")
    )
)

dim_motorista.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_motorista")

# =========================================================
# DIM_ROTA
# =========================================================

rotas = spark.table("silver.Rota")

dim_rota = (
    rotas
    .select(
        monotonically_increasing_id().alias("sk_rota"),

        col("id_rota").alias("id_rota_origem"),

        col("cidade_origem"),
        col("estado_origem"),
        col("cidade_destino"),
        col("estado_destino"),
        col("distancia_km"),

        when(col("distancia_km") < 150, "Curta")
        .when(col("distancia_km") < 400, "Media")
        .otherwise("Longa")
        .alias("faixa_distancia"),

        when(col("estado_origem") == col("estado_destino"), "Estadual")
        .otherwise("Interestadual")
        .alias("tipo_rota")
    )
)

dim_rota.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_rota")

# =========================================================
# DIM_TIPO_CARGA
# =========================================================

tipos_carga = spark.table("silver.TipoCarga")

dim_tipo_carga = (
    tipos_carga
    .select(
        monotonically_increasing_id().alias("sk_tipo_carga"),

        col("id_tipo_carga").alias("id_tipo_carga_origem"),

        col("descricao_tipo_carga"),
        col("categoria")
    )
)

dim_tipo_carga.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_tipo_carga")

# =========================================================
# DIM_DATA
# =========================================================

datas = spark.sql("""
SELECT explode(
    sequence(
        to_date('2022-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )
) AS data_ref
""")

dim_data = (
    datas
    .select(
        date_format(col("data_ref"), "yyyyMMdd")
            .cast("int")
            .alias("sk_data"),

        col("data_ref").alias("data_completa"),

        dayofmonth(col("data_ref")).alias("dia"),

        date_format(col("data_ref"), "EEEE")
            .alias("nome_dia_semana"),

        weekofyear(col("data_ref")).alias("semana_ano"),

        month(col("data_ref")).alias("mes"),

        date_format(col("data_ref"), "MMMM")
            .alias("nome_mes"),

        quarter(col("data_ref")).alias("trimestre"),

        when(month(col("data_ref")) <= 6, 1)
        .otherwise(2)
        .alias("semestre"),

        year(col("data_ref")).alias("ano"),

        when(
            dayofweek(col("data_ref")).isin(1,7),
            1
        )
        .otherwise(0)
        .alias("fim_semana")
    )
)

dim_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_data")

# =========================================================
# FATO_ENTREGA
# =========================================================

entregas = spark.table("silver.Entrega")
rotas = spark.table("silver.Rota")

dim_cliente = spark.table("gold.dim_cliente")
dim_centro = spark.table("gold.dim_centro")
dim_veiculo = spark.table("gold.dim_veiculo")
dim_motorista = spark.table("gold.dim_motorista")
dim_rota = spark.table("gold.dim_rota")
dim_tipo_carga = spark.table("gold.dim_tipo_carga")

fato_entrega = (
    entregas.alias("en")

    .join(
        dim_cliente.alias("dc"),
        col("en.cliente_id") == col("dc.id_cliente_origem")
    )

    .join(
        dim_centro.alias("dce"),
        col("en.centro_id") == col("dce.id_centro_origem")
    )

    .join(
        dim_veiculo.alias("dv"),
        col("en.veiculo_id") == col("dv.id_veiculo_origem")
    )

    .join(
        dim_motorista.alias("dm"),
        col("en.motorista_id") == col("dm.id_motorista_origem")
    )

    .join(
        dim_rota.alias("dr"),
        col("en.rota_id") == col("dr.id_rota_origem")
    )

    .join(
        dim_tipo_carga.alias("dtc"),
        col("en.tipo_carga_id") == col("dtc.id_tipo_carga_origem")
    )

    .join(
        rotas.alias("r"),
        col("en.rota_id") == col("r.id_rota")
    )

    .select(
        col("dc.sk_cliente"),
        col("dce.sk_centro"),
        col("dv.sk_veiculo"),
        col("dm.sk_motorista"),
        col("dr.sk_rota"),
        col("dtc.sk_tipo_carga"),

        date_format(col("en.data_saida"), "yyyyMMdd")
            .cast("int")
            .alias("sk_data_saida"),

        date_format(col("en.data_entrega"), "yyyyMMdd")
            .cast("int")
            .alias("sk_data_entrega"),

        col("en.id_entrega").alias("id_entrega_origem"),

        col("en.status_entrega"),

        col("en.peso_carga_kg"),
        col("en.quantidade_volumes"),

        col("en.valor_frete"),
        col("en.custo_combustivel"),

        (col("en.valor_frete") - col("en.custo_combustivel"))
            .alias("margem_frete"),

        col("r.distancia_km"),

        when(
            col("r.distancia_km") > 0,
            col("en.custo_combustivel") / col("r.distancia_km")
        )
        .otherwise(lit(None))
        .cast("decimal(10,2)")
        .alias("custo_por_km"),

        col("en.tempo_estimado_horas"),
        col("en.tempo_real_horas"),

        (col("en.tempo_real_horas") - col("en.tempo_estimado_horas"))
            .alias("atraso_horas"),

        col("en.avaria"),
        col("en.entrega_no_prazo")
    )
)

fato_entrega.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fato_entrega")

print("Camada GOLD criada com sucesso.")

In [ ]:
%sql
select * from gold.fato_entrega limit 10